# Rubber Band Gun AI - environment check

비공개 GitHub 저장소를 안전하게 내려받고, Colab 의존성과 Google Drive 작업 공간을 확인합니다. 이 노트북은 학습이나 Raspberry Pi 제어를 수행하지 않습니다.

## 1. GitHub Secret 준비

Colab 왼쪽의 열쇠 아이콘에서 `GITHUB_TOKEN`을 추가합니다. `kw-physical-ai/rubber-band-gun-ai` 저장소 읽기 권한만 가진 fine-grained token을 사용합니다.

In [ ]:
import io
import shutil
import zipfile
from pathlib import Path

import requests
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN is required")

headers = {"Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
commit_response = requests.get(
    "https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/commits/main",
    headers=headers,
    timeout=60,
)
commit_response.raise_for_status()
commit_sha = commit_response.json()["sha"]

response = requests.get(
    f"https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/zipball/{commit_sha}",
    headers=headers,
    timeout=60,
)
response.raise_for_status()

extract_root = Path("/content/rubber-band-gun-ai-download")
repo_root = Path("/content/rubber-band-gun-ai")
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(repo_root, ignore_errors=True)

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    archive.extractall(extract_root)
downloaded_root = next(extract_root.iterdir())
shutil.move(str(downloaded_root), repo_root)
del token, headers, response, commit_response
print(f"Repository ready: {repo_root} at {commit_sha}")

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_root / "requirements/colab.txt")]
)
sys.path.insert(0, str(repo_root / "src"))
print("Colab dependencies installed")

In [ ]:
import json

from rubber_band_gun import runtime_summary

summary = runtime_summary()
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 2. Google Drive 작업 공간

최초 실행 시 Drive 접근을 승인합니다. `processed/`와 `runs/`는 Colab이 만들고, `pi-upload/raw/`와 `pi-upload/smoke/`는 제한된 권한의 Pi 업로더가 만듭니다.

In [ ]:
from google.colab import drive

from rubber_band_gun import ProjectWorkspace

drive.mount("/content/drive")
workspace = ProjectWorkspace(Path("/content/drive/MyDrive/rubber-band-gun-ai-data"))
workspace.create()
print(f"raw: {workspace.raw}")
print(f"processed: {workspace.processed}")
print(f"smoke: {workspace.smoke}")
print(f"runs: {workspace.runs}")

## 3. Pi → Drive → Colab smoke bundle 검증

Pi가 출력한 capture ID를 입력하면 `pi-upload/smoke/`의 해당 번들 파일 수와 SHA-256을 확인하고 `runs/data-flow-smoke/`에 보고서를 저장합니다. 이 이미지는 연결 검증 전용이며 학습이나 모델 평가에 사용하지 않습니다.

In [ ]:
from datetime import UTC, datetime

from rubber_band_gun import verify_smoke_bundle

capture_id = input("Capture ID printed by the Pi: ").strip()
bundle = workspace.smoke / capture_id
if not bundle.is_dir():
    raise FileNotFoundError(f"Smoke bundle not found: {bundle}")
report = verify_smoke_bundle(bundle)
report["colab_commit"] = commit_sha
report["colab_verified_at_utc"] = datetime.now(UTC).isoformat()
report_root = workspace.runs / "data-flow-smoke"
report_root.mkdir(parents=True, exist_ok=True)
report_path = report_root / f"{report['capture_id']}.json"
report_json = json.dumps(report, ensure_ascii=False, indent=2) + "\n"
report_path.write_text(report_json, encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print(f"Smoke report saved: {report_path}")